# IFRS S1/S2 Evidence-Grounded Report Generation Pipeline

Generates a five-section IFRS S1/S2 sustainability disclosure report (General Requirements, Governance, Strategy, Risk Management, Metrics and Targets) from synthetic bank data, with Azure OpenAI (`gpt-5.1` + `gpt-5.2`) orchestrated by LangGraph.

**Controlling invariant.** *Missing data is a property of the evidence, discovered deterministically, and carried in a separate channel that no writer node ever reads.* The disclosable channel and the gap channel diverge once, deterministically (§5), before any prompt is built — so there is no code path by which an unsupported requirement reaches the writer. This yields the three required properties **structurally**:

1. **Missing data is audit-only** — recorded in the audit bundle / gap report, never in the report.
2. **Missing data cannot lower scores** — coverage uses the *disclosable denominator* (§8).
3. **Missing data never appears in text** — structural + writer prompt + deterministic forbidden-phrase gate (§7).

**Style-guide conflict (resolved, §6).** The supplied style guide's `missing_data_language_rules` and status labels ("not currently available", "under development", "not reported for the period") contradict the audit-only rule. Resolved in favour of the controlling requirement: those rules are excluded from the style rubric and their phrases are added to the forbidden-phrase gate.

Run the notebook top to bottom. The final cell runs end-to-end on the uploaded data using a deterministic offline `MockLLM`; switch one line to `AzureOpenAILLM()` for live generation. The deterministic validators (number grounding, phrase filter, coverage denominator, numeric reconciliation, assembly, audit) run for real either way.

## 0. Dependencies

In [1]:
# Run once. In a fresh environment, uncomment:
# %pip install -q langgraph langchain-core pydantic openai
import json, re, math, hashlib, types, os, sys
from pathlib import Path
from datetime import date, datetime, timezone
from enum import Enum
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Annotated, Any, Literal, Optional, Protocol, TypedDict, runtime_checkable
from pydantic import BaseModel, Field
from langgraph.graph import END, START, StateGraph
print('imports ok')

imports ok


## 1. Configuration
Model deployments by role, gate thresholds, and canonical section order. Roles map to deployments here, so swapping `gpt-5.1` / `gpt-5.2` is a one-line change.

In [2]:
"""Central configuration.

All tunable policy lives here so the rest of the package contains no magic
numbers. Two Azure OpenAI deployments are referenced by role, never by name,
inside the nodes -- so swapping gpt-5.1 / gpt-5.2 is a one-line change.
"""

import os
from dataclasses import dataclass, field
from typing import Literal

Role = Literal["writer", "extractor", "judge", "reviser"]


@dataclass(frozen=True)
class ModelConfig:
    """Maps a logical role to an Azure deployment + sampling policy.

    Rationale: the costly failure in this pipeline is a *false pass* (a
    hallucination slipping through a judge), not a slow draft. So the stronger
    model goes on the safety-critical side (judge, reviser) and a *different*
    model judges than writes, to reduce self-preference bias.
    """

    writer_deployment: str = os.getenv("AZURE_WRITER_DEPLOYMENT", "gpt-5.1")
    extractor_deployment: str = os.getenv("AZURE_EXTRACTOR_DEPLOYMENT", "gpt-5.1")
    judge_deployment: str = os.getenv("AZURE_JUDGE_DEPLOYMENT", "gpt-5.2")
    reviser_deployment: str = os.getenv("AZURE_REVISER_DEPLOYMENT", "gpt-5.2")

    writer_temperature: float = 0.3
    extractor_temperature: float = 0.0
    judge_temperature: float = 0.0
    reviser_temperature: float = 0.0

    def deployment_for(self, role: Role) -> str:
        return getattr(self, f"{role}_deployment")

    def temperature_for(self, role: Role) -> float:
        return getattr(self, f"{role}_temperature")


@dataclass(frozen=True)
class Thresholds:
    """Gate and scoring policy.

    integrity_min is a hard gate (~1.0): every claim entailed, every number
    grounded. style_min and coverage are *reported* scores, never traded
    against integrity.
    """

    integrity_min: float = 1.0          # hard gate: no ungrounded numbers / claims
    style_min: float = 0.70             # soft gate: triggers revision if below
    max_revisions: int = 3              # bounded loop; then human-review flag
    numeric_rel_tolerance: float = 0.005  # 0.5% relative tolerance for number match
    numeric_abs_tolerance: float = 1e-6


@dataclass(frozen=True)
class Settings:
    model: ModelConfig = field(default_factory=ModelConfig)
    thresholds: Thresholds = field(default_factory=Thresholds)
    # Canonical IFRS pillar ordering used everywhere (assembly, iteration).
    section_order: tuple[str, ...] = (
        "general_requirements",
        "governance",
        "strategy",
        "risk_management",
        "metrics_and_targets",
    )
    section_titles: dict[str, str] = field(
        default_factory=lambda: {
            "general_requirements": "General Requirements",
            "governance": "Governance",
            "strategy": "Strategy",
            "risk_management": "Risk Management",
            "metrics_and_targets": "Metrics and Targets",
        }
    )

    def title_for(self, section_key: str) -> str:
        return self.section_titles.get(section_key, section_key)


SETTINGS = Settings()

## 2. Contracts & State
The single source of truth for every artifact that crosses a layer boundary: `RequirementBinding` (Layer 1→2 contract), `ValidationReport`, `SectionScore`, `AuditBundle`, and the LangGraph `GraphState`. Note `RequirementBinding.is_disclosable` and the `integrity_score` / `hard_gates_pass` gate logic.

In [3]:
"""State and contract models.

This module is the single source of truth for every artifact that crosses a
layer boundary. The design mirrors a contract-boundary chain:

    Layer 1 (deterministic ingestion/binding)
        --> RequirementBinding[]  (THE contract artifact)
    Layer 2 (LangGraph generation)
        --> SectionState[]  (drafts + validation + scores)
    Layer 3 (deterministic assembly/audit)
        --> final report + AuditBundle

The central invariant (everything else follows from it):

    Missing data is a property of the evidence, discovered deterministically,
    and carried in a SEPARATE channel (`audit`) that no writer node ever reads.

So `disclosable_bindings` (what writers see) and `gap_ledger` (audit-only)
diverge ONCE, deterministically, before any LLM prompt is built. There is no
code path by which a MISSING requirement reaches the writer.
"""

from enum import Enum
from typing import Annotated, Any, Optional, TypedDict

from pydantic import BaseModel, Field


# --------------------------------------------------------------------------- #
# Enumerations
# --------------------------------------------------------------------------- #
class SupportStatus(str, Enum):
    SUPPORTED = "SUPPORTED"
    PARTIAL = "PARTIAL"
    MISSING = "MISSING"


class ClaimVerdict(str, Enum):
    ENTAILED = "ENTAILED"
    NOT_ENTAILED = "NOT_ENTAILED"
    CONTRADICTED = "CONTRADICTED"


class SectionStatus(str, Enum):
    PENDING = "PENDING"
    DRAFTED = "DRAFTED"
    ACCEPTED = "ACCEPTED"
    NEEDS_HUMAN_REVIEW = "NEEDS_HUMAN_REVIEW"


# --------------------------------------------------------------------------- #
# Inputs (immutable once loaded)
# --------------------------------------------------------------------------- #
class Requirement(BaseModel):
    """One IFRS S1/S2 datapoint. Mirrors the uploaded requirement schema."""

    requirement_id: str
    standard: str
    paragraph_id: str
    report_section: str
    requirement_text: str
    clause_path: Optional[str] = None
    obligation_type: str = "mandatory"
    mandatory: bool = True
    evidence_tags: list[str] = Field(default_factory=list)
    banking_relevance: Optional[str] = None
    # Weight used in coverage scoring. Mandatory requirements weigh more.
    requirement_quality_score: float = 1.0


class EvidenceRef(BaseModel):
    """A pointer into the payload, kept as a path so it is reviewable/traceable."""

    path: str                       # e.g. "governance[0].esg_committee_exists"
    value: Any = None               # the resolved value, captured at bind time
    source_table: Optional[str] = None  # top-level payload key, e.g. "board_minutes"


# --------------------------------------------------------------------------- #
# Layer 1 output: the binding contract
# --------------------------------------------------------------------------- #
class RequirementBinding(BaseModel):
    """THE contract artifact between Layer 1 and Layer 2.

    Produced deterministically (tag registry + retrieval), with an LLM only
    *confirming* sufficiency. Persisted verbatim into the audit bundle, so the
    requirement->evidence mapping is reviewable line by line rather than a
    black box.
    """

    requirement_id: str
    section_key: str
    status: SupportStatus
    evidence_refs: list[EvidenceRef] = Field(default_factory=list)
    covered_elements: list[str] = Field(default_factory=list)
    missing_elements: list[str] = Field(default_factory=list)
    rationale: str = ""
    weight: float = 1.0

    @property
    def is_disclosable(self) -> bool:
        """SUPPORTED and PARTIAL contribute to the report; MISSING never does."""
        return self.status in (SupportStatus.SUPPORTED, SupportStatus.PARTIAL)


# --------------------------------------------------------------------------- #
# Validation artifacts (per section)
# --------------------------------------------------------------------------- #
class NumericCheck(BaseModel):
    surface: str            # the literal token found in the draft, e.g. "1,154.8"
    normalized: float
    unit: Optional[str] = None
    grounded: bool = False
    matched_path: Optional[str] = None
    matched_value: Optional[float] = None


class ClaimCheck(BaseModel):
    claim: str
    verdict: ClaimVerdict
    evidence_ref: Optional[str] = None
    note: str = ""


class PhraseHit(BaseModel):
    phrase: str
    context: str            # surrounding text for the auditor


class StyleScore(BaseModel):
    dimensions: dict[str, float] = Field(default_factory=dict)  # tone, structure, ...
    overall: float = 0.0
    notes: str = ""


class ValidationReport(BaseModel):
    """Structured outcome of the validator battery for one section/draft."""

    numeric_checks: list[NumericCheck] = Field(default_factory=list)
    claim_checks: list[ClaimCheck] = Field(default_factory=list)
    phrase_hits: list[PhraseHit] = Field(default_factory=list)
    style: StyleScore = Field(default_factory=StyleScore)

    # --- derived gate signals ------------------------------------------------
    @property
    def numbers_ok(self) -> bool:
        return all(c.grounded for c in self.numeric_checks)

    @property
    def claims_ok(self) -> bool:
        return all(c.verdict == ClaimVerdict.ENTAILED for c in self.claim_checks)

    @property
    def phrases_ok(self) -> bool:
        return len(self.phrase_hits) == 0

    @property
    def integrity_score(self) -> float:
        """Fraction of claims entailed AND numbers grounded. Hard gate ~1.0."""
        total = len(self.numeric_checks) + len(self.claim_checks)
        if total == 0:
            return 1.0
        good = sum(c.grounded for c in self.numeric_checks)
        good += sum(c.verdict == ClaimVerdict.ENTAILED for c in self.claim_checks)
        return good / total

    def hard_gates_pass(self, style_min: float) -> bool:
        return (
            self.numbers_ok
            and self.claims_ok
            and self.phrases_ok
            and self.style.overall >= style_min
        )

    def failure_summary(self) -> list[str]:
        """Structured, specific feedback handed to the reviser (never 'try again')."""
        out: list[str] = []
        for c in self.numeric_checks:
            if not c.grounded:
                out.append(
                    f"UNGROUNDED_NUMBER: '{c.surface}' has no match in the evidence. "
                    "Remove it or replace it with an evidence-backed value."
                )
        for c in self.claim_checks:
            if c.verdict != ClaimVerdict.ENTAILED:
                out.append(
                    f"{c.verdict.value}_CLAIM: \"{c.claim}\" -- {c.note}. "
                    "Remove or correct to match the evidence; do not introduce new facts."
                )
        for p in self.phrase_hits:
            out.append(
                f"FORBIDDEN_PHRASE: '{p.phrase}' (in \"...{p.context}...\"). "
                "Delete any reference to absent, unavailable, or pending information."
            )
        if self.style.overall and self.style.notes:
            out.append(f"STYLE: {self.style.notes}")
        return out


# --------------------------------------------------------------------------- #
# Scoring
# --------------------------------------------------------------------------- #
class SectionScore(BaseModel):
    """Three INDEPENDENT sub-scores, never collapsed until report level.

    coverage uses a DISCLOSABLE denominator: MISSING requirements are removed
    before the ratio is computed, so synthetic gaps cannot lower the score.
    """

    coverage: float = 0.0
    integrity: float = 0.0
    style: float = 0.0
    disclosable_count: int = 0
    total_requirements: int = 0
    excluded_missing: int = 0


# --------------------------------------------------------------------------- #
# Layer 2 working state (per section)
# --------------------------------------------------------------------------- #
class SectionState(BaseModel):
    section_key: str
    title: str
    draft: str = ""
    status: SectionStatus = SectionStatus.PENDING
    revision_count: int = 0
    validation: Optional[ValidationReport] = None
    score: Optional[SectionScore] = None
    revision_log: list[str] = Field(default_factory=list)


# --------------------------------------------------------------------------- #
# Audit channel (parallel, write-only from the writers' perspective)
# --------------------------------------------------------------------------- #
class GapEntry(BaseModel):
    requirement_id: str
    section_key: str
    status: SupportStatus           # MISSING, or PARTIAL (uncovered elements)
    missing_elements: list[str] = Field(default_factory=list)
    reason: str = ""


class AuditBundle(BaseModel):
    binding_ledger: list[RequirementBinding] = Field(default_factory=list)
    gap_ledger: list[GapEntry] = Field(default_factory=list)
    claim_verdicts: dict[str, list[ClaimCheck]] = Field(default_factory=dict)
    numeric_ledger: dict[str, list[NumericCheck]] = Field(default_factory=dict)
    revision_history: dict[str, list[str]] = Field(default_factory=dict)
    run_metadata: dict[str, Any] = Field(default_factory=dict)


# --------------------------------------------------------------------------- #
# LangGraph state (the TypedDict that flows between nodes)
# --------------------------------------------------------------------------- #
def _merge_dicts(a: dict, b: dict) -> dict:
    """Reducer so concurrent section branches can write disjoint keys safely."""
    out = dict(a or {})
    out.update(b or {})
    return out


class GraphState(TypedDict, total=False):
    # Inputs ----------------------------------------------------------------
    requirements: dict[str, list[Requirement]]   # section_key -> requirements
    payloads: dict[str, dict]                     # section_key -> raw payload
    style_guide: dict
    bank_id: str

    # Layer 1 output --------------------------------------------------------
    bindings: dict[str, list[RequirementBinding]]  # section_key -> bindings
    disclosable: dict[str, list[RequirementBinding]]  # section_key -> disclosable

    # Layer 2 working state (fan-out writes one section key each) -----------
    sections: Annotated[dict[str, SectionState], _merge_dicts]

    # Layer 3 ---------------------------------------------------------------
    final_report: str
    consistency_report: dict
    report_score: dict

    # Parallel audit channel (NEVER read by a writer node) ------------------
    audit: AuditBundle

## 3. Prompts
Versioned, hashable prompt templates. Each carries a `TASK:` tag so any client (real or mock) routes deterministically. The writer prompt encodes the no-missing-data-language rule. After definition we expose them through a `T` namespace used by later sections.

In [4]:
"""Prompt templates.

One module so every prompt is versioned and hashable for the audit trail.
Each prompt opens with a `TASK:` tag; this lets any client route deterministically
and lets the audit log record exactly which instruction produced an artifact.

KEY POLICY ENCODED IN THE WRITER PROMPT
---------------------------------------
The writer is given ONLY disclosable evidence. It is additionally instructed to
never reference the absence, sufficiency, completeness, or provenance of
information. This is defense-in-depth: the structural guarantee (no MISSING
requirement is ever passed in) is primary; the instruction is the second layer;
the deterministic forbidden-phrase gate is the third.
"""

import json

# Task tags (single source of truth) ----------------------------------------
TASK_CLASSIFY = "classify_sufficiency"
TASK_WRITE = "write_section"
TASK_EXTRACT_CLAIMS = "extract_claims"
TASK_JUDGE_ENTAILMENT = "judge_entailment"
TASK_JUDGE_STYLE = "judge_style"
TASK_REVISE = "revise_section"
TASK_JUDGE_CONSISTENCY = "judge_consistency"


# --------------------------------------------------------------------------- #
# Layer 1: sufficiency classifier
# --------------------------------------------------------------------------- #
CLASSIFIER_SYSTEM = (
    "You are an IFRS S1/S2 disclosure analyst. You decide whether the supplied "
    "evidence is sufficient to satisfy a specific disclosure requirement. You are "
    "conservative: absence of evidence means MISSING. You never invent support. "
    "Respond with a single JSON object and nothing else."
)


def classifier_user(requirement_text: str, evidence_json: str) -> str:
    return (
        f"TASK: {TASK_CLASSIFY}\n\n"
        "Decide the support status of the requirement given ONLY the evidence below.\n"
        "Return JSON: {\"status\": \"SUPPORTED|PARTIAL|MISSING\", "
        "\"covered_elements\": [..], \"missing_elements\": [..], \"rationale\": \"..\"}\n"
        "- SUPPORTED: evidence covers every disclosable element of the requirement.\n"
        "- PARTIAL: evidence covers some elements; list the rest in missing_elements.\n"
        "- MISSING: no relevant evidence.\n\n"
        f"REQUIREMENT:\n{requirement_text}\n\n"
        f"EVIDENCE (JSON):\n{evidence_json}\n"
    )


# --------------------------------------------------------------------------- #
# Layer 2: writer
# --------------------------------------------------------------------------- #
WRITER_SYSTEM = (
    "You are a sustainability-disclosure writer producing audit-ready IFRS S1/S2 "
    "report sections. Hard rules:\n"
    "1. Disclose ONLY what the supplied evidence supports. Every number and named "
    "fact must come from the evidence.\n"
    "2. Never reference the absence, unavailability, incompleteness, sufficiency, "
    "or provenance of information. Do not use status labels such as 'not "
    "available', 'under development', or 'not reported'. If something is not in "
    "the evidence, simply do not mention it.\n"
    "3. Follow the supplied style guide for voice, structure, and terminology.\n"
    "4. Do not invent committee names, dates, vendors, or figures."
)


def writer_user(
    section_title: str,
    style_rules_json: str,
    requirement_briefs: str,
    evidence_json: str,
) -> str:
    return (
        f"TASK: {TASK_WRITE}\n\n"
        f"Write the '{section_title}' section of an IFRS S1/S2 sustainability "
        "disclosure report.\n\n"
        f"STYLE RULES (JSON):\n{style_rules_json}\n\n"
        f"REQUIREMENTS TO ADDRESS (only these):\n{requirement_briefs}\n\n"
        f"EVIDENCE (JSON, the ONLY facts you may state):\n{evidence_json}\n"
    )


# --------------------------------------------------------------------------- #
# Layer 2: claim extraction + entailment + style judges
# --------------------------------------------------------------------------- #
CLAIM_EXTRACT_SYSTEM = (
    "You decompose disclosure prose into atomic, independently checkable factual "
    "claims. Respond with a single JSON object and nothing else."
)


def claim_extract_user(draft: str) -> str:
    return (
        f"TASK: {TASK_EXTRACT_CLAIMS}\n\n"
        "Split the text into atomic factual claims (one verifiable assertion each).\n"
        "Exclude pure transitions and headings.\n"
        "Return JSON: {\"claims\": [\"..\", \"..\"]}\n\n"
        f"TEXT:\n{draft}\n"
    )


ENTAILMENT_SYSTEM = (
    "You are a strict factual auditor. For each claim you decide whether the "
    "evidence ENTAILS it, does NOT entail it, or CONTRADICTS it. A claim that is "
    "plausible but not present in the evidence is NOT_ENTAILED. Respond with a "
    "single JSON object and nothing else."
)


def entailment_user(claims_json: str, evidence_json: str) -> str:
    return (
        f"TASK: {TASK_JUDGE_ENTAILMENT}\n\n"
        "For each claim return a verdict against the evidence.\n"
        "Return JSON: {\"results\": [{\"claim\": \"..\", "
        "\"verdict\": \"ENTAILED|NOT_ENTAILED|CONTRADICTED\", "
        "\"evidence_ref\": \"path or null\", \"note\": \"..\"}]}\n\n"
        f"CLAIMS (JSON):\n{claims_json}\n\n"
        f"EVIDENCE (JSON):\n{evidence_json}\n"
    )


STYLE_SYSTEM = (
    "You are a disclosure style reviewer. You score prose against an explicit "
    "rubric on a 0-1 scale per dimension. You do NOT reward or penalise mentions "
    "of missing data; that dimension is out of scope. Respond with a single JSON "
    "object and nothing else."
)


def style_user(draft: str, rubric_json: str) -> str:
    return (
        f"TASK: {TASK_JUDGE_STYLE}\n\n"
        "Score the text on each rubric dimension (0-1) and give one short note on "
        "the weakest dimension.\n"
        "Return JSON: {\"dimensions\": {\"tone\": 0-1, \"structure\": 0-1, "
        "\"ifrs_register\": 0-1, \"conciseness\": 0-1}, \"notes\": \"..\"}\n\n"
        f"RUBRIC (JSON):\n{rubric_json}\n\n"
        f"TEXT:\n{draft}\n"
    )


# --------------------------------------------------------------------------- #
# Layer 2: reviser
# --------------------------------------------------------------------------- #
REVISER_SYSTEM = (
    "You repair an IFRS disclosure draft. You may ONLY remove or correct content "
    "to fix the listed problems. You must NOT introduce any new fact, number, or "
    "claim. The same hard rules as the writer apply: never reference missing or "
    "unavailable information."
)


def reviser_user(draft: str, failures: str, evidence_json: str) -> str:
    return (
        f"TASK: {TASK_REVISE}\n\n"
        "Revise the draft to fix EVERY listed problem. Remove unsupported content "
        "rather than inventing support. Return only the corrected section text.\n\n"
        f"PROBLEMS TO FIX:\n{failures}\n\n"
        f"EVIDENCE (JSON, the ONLY facts you may state):\n{evidence_json}\n\n"
        f"DRAFT:\n{draft}\n"
    )


# --------------------------------------------------------------------------- #
# Layer 3: cross-section coherence judge
# --------------------------------------------------------------------------- #
CONSISTENCY_SYSTEM = (
    "You check a multi-section disclosure report for narrative and terminological "
    "consistency (e.g. a governance body named consistently, no contradictory "
    "statements). You do not re-check numbers. Respond with a single JSON object."
)


def consistency_user(report_text: str) -> str:
    return (
        f"TASK: {TASK_JUDGE_CONSISTENCY}\n\n"
        "Identify cross-section inconsistencies in terminology or narrative.\n"
        "Return JSON: {\"inconsistencies\": [{\"sections\": [\"..\"], "
        "\"issue\": \"..\"}]}\n\n"
        f"REPORT:\n{report_text}\n"
    )


def to_json(obj: object) -> str:
    return json.dumps(obj, ensure_ascii=False, indent=2, default=str)

# Namespace handle used throughout the notebook (mirrors the package's `prompts.templates`).
T = types.SimpleNamespace(
    TASK_CLASSIFY=TASK_CLASSIFY, TASK_WRITE=TASK_WRITE, TASK_EXTRACT_CLAIMS=TASK_EXTRACT_CLAIMS,
    TASK_JUDGE_ENTAILMENT=TASK_JUDGE_ENTAILMENT, TASK_JUDGE_STYLE=TASK_JUDGE_STYLE,
    TASK_REVISE=TASK_REVISE, TASK_JUDGE_CONSISTENCY=TASK_JUDGE_CONSISTENCY,
    CLASSIFIER_SYSTEM=CLASSIFIER_SYSTEM, classifier_user=classifier_user,
    WRITER_SYSTEM=WRITER_SYSTEM, writer_user=writer_user,
    CLAIM_EXTRACT_SYSTEM=CLAIM_EXTRACT_SYSTEM, claim_extract_user=claim_extract_user,
    ENTAILMENT_SYSTEM=ENTAILMENT_SYSTEM, entailment_user=entailment_user,
    STYLE_SYSTEM=STYLE_SYSTEM, style_user=style_user,
    REVISER_SYSTEM=REVISER_SYSTEM, reviser_user=reviser_user,
    CONSISTENCY_SYSTEM=CONSISTENCY_SYSTEM, consistency_user=consistency_user,
    to_json=to_json,
)

## 4. LLM Clients
A role-based `LLMClient` protocol, the Azure-backed implementation (`gpt-5.1` / `gpt-5.2`, with retry), and a deterministic `MockLLM`. The mock composes prose ONLY from evidence values embedded in the prompt, so the real deterministic validators run honestly against it; it only stands in for soft LLM judgement.

In [5]:
"""LLM client protocol.

Nodes depend on this Protocol, never on a concrete SDK. That keeps the graph
testable offline (MockLLM) and makes the Azure deployment swap a config change.
Every call is role-tagged ('writer' | 'extractor' | 'judge' | 'reviser') so the
client resolves the right deployment and temperature from config.
"""

from typing import Any, Protocol, runtime_checkable


@runtime_checkable
class LLMClient(Protocol):
    def complete(
        self,
        *,
        role: Role,
        system: str,
        user: str,
        json_mode: bool = False,
    ) -> str:
        """Return the model's text completion (raw string).

        When json_mode is True the client requests structured JSON output and
        the caller is responsible for parsing. Implementations must be
        deterministic for a given (role, system, user) when temperature == 0.
        """
        ...


def parse_json_strict(text: str) -> Any:
    """Parse model JSON robustly: strip code fences, locate the JSON body.

    Used by every structured-output call (classifier, claim/number extraction,
    judges) so parsing logic lives in exactly one place.
    """
    import json

    cleaned = text.strip()
    if cleaned.startswith("```"):
        # remove ```json ... ``` fences
        cleaned = cleaned.split("```", 2)
        cleaned = cleaned[1] if len(cleaned) >= 2 else text
        if cleaned.lstrip().lower().startswith("json"):
            cleaned = cleaned.lstrip()[4:]
    cleaned = cleaned.strip().strip("`").strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        # last resort: extract the outermost {...} or [...]
        start = min(
            (i for i in (cleaned.find("{"), cleaned.find("[")) if i != -1),
            default=-1,
        )
        end = max(cleaned.rfind("}"), cleaned.rfind("]"))
        if start != -1 and end != -1 and end > start:
            return json.loads(cleaned[start : end + 1])
        raise


"""Azure OpenAI implementation of LLMClient.

Wires the role->deployment mapping from config to the Azure OpenAI SDK. Adds
retry/backoff because the dangerous production failure mode here is a transient
error aborting a long multi-section run. No business logic lives here.
"""

import os
import time


class AzureOpenAILLM(LLMClient):
    def __init__(
        self,
        *,
        endpoint: str | None = None,
        api_key: str | None = None,
        api_version: str = "2024-10-21",
        max_retries: int = 4,
    ) -> None:
        # Imported lazily so the package imports without the SDK present
        # (e.g. when running the offline MockLLM demo).
        from openai import AzureOpenAI

        self._client = AzureOpenAI(
            azure_endpoint=endpoint or os.environ["AZURE_OPENAI_ENDPOINT"],
            api_key=api_key or os.environ["AZURE_OPENAI_API_KEY"],
            api_version=api_version,
        )
        self._max_retries = max_retries
        self._cfg = SETTINGS.model

    def complete(
        self,
        *,
        role: Role,
        system: str,
        user: str,
        json_mode: bool = False,
    ) -> str:
        deployment = self._cfg.deployment_for(role)
        temperature = self._cfg.temperature_for(role)
        kwargs: dict = {
            "model": deployment,
            "temperature": temperature,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        }
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}

        last_err: Exception | None = None
        for attempt in range(self._max_retries):
            try:
                resp = self._client.chat.completions.create(**kwargs)
                return resp.choices[0].message.content or ""
            except Exception as err:  # noqa: BLE001 - SDK raises many types
                last_err = err
                time.sleep(min(2**attempt, 8))
        raise RuntimeError(f"Azure OpenAI call failed after retries: {last_err}")


"""Deterministic mock LLM for offline runs, demos, and CI.

It produces schema-valid, self-consistent outputs by reading the SAME prompts a
real model would receive and routing on the `TASK:` tag. Crucially, the writer
mock composes prose ONLY from evidence values embedded in the prompt, so the
*real* deterministic validators (numeric grounding, forbidden-phrase filter) run
for real against it. The mock only stands in for genuine LLM judgement
(entailment, style), which it grants when the deterministic checks would pass.

This means the load-bearing guarantees of the pipeline are exercised honestly in
the offline demo; only the soft judgements are simulated.
"""

import json
import re
from typing import Any


def _extract_block(user: str, header: str) -> str:
    """Pull a `HEADER (JSON):\n{...}` block out of a prompt body."""
    idx = user.find(header)
    if idx == -1:
        return "{}"
    tail = user[idx + len(header):]
    brace = tail.find("{")
    bracket = tail.find("[")
    start = min([p for p in (brace, bracket) if p != -1], default=-1)
    if start == -1:
        return "{}"
    return _balanced(tail[start:])


def _balanced(s: str) -> str:
    open_c = s[0]
    close_c = "}" if open_c == "{" else "]"
    depth, in_str, esc = 0, False, False
    for i, ch in enumerate(s):
        if esc:
            esc = False
            continue
        if ch == "\\":
            esc = True
            continue
        if ch == '"':
            in_str = not in_str
        elif not in_str:
            if ch == open_c:
                depth += 1
            elif ch == close_c:
                depth -= 1
                if depth == 0:
                    return s[: i + 1]
    return s


def _flatten(obj: Any, prefix: str = "") -> dict[str, Any]:
    out: dict[str, Any] = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            out.update(_flatten(v, f"{prefix}.{k}" if prefix else str(k)))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            out.update(_flatten(v, f"{prefix}[{i}]"))
    else:
        out[prefix] = obj
    return out


class MockLLM(LLMClient):
    """Routes on TASK tag; deterministic given identical prompts."""

    def complete(self, *, role: Role, system: str, user: str, json_mode: bool = False) -> str:
        if T.TASK_CLASSIFY in user:
            return self._classify(user)
        if T.TASK_WRITE in user:
            return self._write(user)
        if T.TASK_EXTRACT_CLAIMS in user:
            return self._extract_claims(user)
        if T.TASK_JUDGE_ENTAILMENT in user:
            return self._entail(user)
        if T.TASK_JUDGE_STYLE in user:
            return self._style(user)
        if T.TASK_REVISE in user:
            return self._revise(user)
        if T.TASK_JUDGE_CONSISTENCY in user:
            return json.dumps({"inconsistencies": []})
        return ""

    # --- classifier: SUPPORTED iff evidence non-empty -----------------------
    def _classify(self, user: str) -> str:
        ev = parse_json_strict(_extract_block(user, "EVIDENCE (JSON):"))
        leaves = {k: v for k, v in _flatten(ev).items() if v not in (None, "", [], {})}
        if not leaves:
            return json.dumps(
                {"status": "MISSING", "covered_elements": [],
                 "missing_elements": ["all elements"], "rationale": "No evidence."}
            )
        # PARTIAL if very thin evidence, else SUPPORTED (deterministic heuristic).
        status = "PARTIAL" if len(leaves) < 2 else "SUPPORTED"
        missing = ["secondary detail"] if status == "PARTIAL" else []
        return json.dumps(
            {"status": status, "covered_elements": list(leaves.keys())[:5],
             "missing_elements": missing, "rationale": "Evidence present in payload."}
        )

    # --- writer: prose built only from evidence leaves ----------------------
    def _write(self, user: str) -> str:
        title = ""
        m = re.search(r"Write the '([^']+)' section", user)
        if m:
            title = m.group(1)
        ev = parse_json_strict(_extract_block(user, "EVIDENCE (JSON, the ONLY facts you may state):"))
        leaves = _flatten(ev)
        # Build sentences from scalar leaves only -> every number is grounded.
        scalars = [
            (k, v) for k, v in leaves.items()
            if isinstance(v, (int, float)) or (isinstance(v, str) and len(v) < 60)
        ]
        lines = [f"## {title}", ""]
        lines.append(
            f"This section sets out the {title.lower()} disclosures supported by "
            "the available evidence for the reporting entity."
        )
        lines.append("")
        for k, v in scalars[:14]:
            label = k.split(".")[-1].replace("_", " ")
            if isinstance(v, bool):
                lines.append(f"- The {label} status is recorded as {str(v).lower()}.")
            elif isinstance(v, (int, float)):
                lines.append(f"- The reported {label} is {v}.")
            else:
                lines.append(f"- The {label} is {v}.")
        lines.append("")
        lines.append(
            "The entity describes the processes and controls applied to the matters "
            "above and cross-references related disclosures in other sections."
        )
        return "\n".join(lines)

    # --- claim extraction: bullet/sentence split ----------------------------
    def _extract_claims(self, user: str) -> str:
        idx = user.find("TEXT:\n")
        text = user[idx + 6:] if idx != -1 else user
        claims = []
        for ln in text.splitlines():
            ln = ln.strip().lstrip("-").strip()
            if not ln or ln.startswith("#"):
                continue
            if len(ln) > 12:
                claims.append(ln)
        return json.dumps({"claims": claims})

    # --- entailment: ENTAILED (mock grants; real deterministic numeric gate
    #     is what actually protects against hallucinated numbers) -----------
    def _entail(self, user: str) -> str:
        claims = parse_json_strict(_extract_block(user, "CLAIMS (JSON):")).get("claims", [])
        results = [
            {"claim": c, "verdict": "ENTAILED", "evidence_ref": "payload", "note": "ok"}
            for c in claims
        ]
        return json.dumps({"results": results})

    # --- style: passing score ----------------------------------------------
    def _style(self, user: str) -> str:
        return json.dumps(
            {"dimensions": {"tone": 0.9, "structure": 0.85, "ifrs_register": 0.88,
                            "conciseness": 0.82}, "notes": "Clear and on-register."}
        )

    # --- reviser: strip lines flagged with forbidden phrases ---------------
    def _revise(self, user: str) -> str:
        idx = user.find("DRAFT:\n")
        draft = user[idx + 7:] if idx != -1 else ""
        return draft  # mock keeps draft; real reviser would correct flagged items

## 5. Evidence — Layer 1 (deterministic ingestion + binding)
Load requirements/payloads/style; the curated `evidence_tag → payload-table` registry; the binder (reviewable `EvidenceRef`s); the sufficiency classifier (deterministic MISSING pre-gate + LLM confirmation); and `split_channels` — **the one place** the disclosable and gap channels diverge.

In [6]:
"""Deterministic loaders (Layer 1).

Reads the uploaded artifacts into typed inputs. No LLM, no network: this layer
must be fully reproducible. The requirement schema is the one observed in the
uploaded *_requirements.json files (S1/S2 split under `standards`).
"""

import json
import math
from pathlib import Path


def _opt_str(value) -> str | None:
    """Coerce JSON NaN (float) / non-string values to None for optional strings."""
    if isinstance(value, str):
        return value
    return None


def _safe_float(value, default: float = 1.0) -> float:
    try:
        f = float(value)
    except (TypeError, ValueError):
        return default
    return default if math.isnan(f) else f


def load_requirements(path: str | Path) -> list[Requirement]:
    """Flatten the {standards: {IFRS S1, IFRS S2}} structure into Requirement[]."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    section_key = data.get("section_key", "")
    out: list[Requirement] = []
    for std_block in data.get("standards", {}).values():
        for raw in std_block.get("requirements", []):
            out.append(
                Requirement(
                    requirement_id=raw["requirement_id"],
                    standard=raw["standard"],
                    paragraph_id=str(raw.get("paragraph_id", "")),
                    report_section=raw.get("report_section", section_key),
                    requirement_text=raw.get("requirement_text", ""),
                    clause_path=_opt_str(raw.get("clause_path")),
                    obligation_type=raw.get("obligation_type", "mandatory"),
                    mandatory=bool(raw.get("mandatory", True)),
                    evidence_tags=list(raw.get("evidence_tags", [])),
                    banking_relevance=_opt_str(raw.get("banking_relevance")),
                    requirement_quality_score=_safe_float(
                        raw.get("requirement_quality_score", 1.0)
                    ),
                )
            )
    return out


def load_payload(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def load_style_guide(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def load_all(
    requirement_paths: dict[str, str | Path],
    payload_paths: dict[str, str | Path],
    style_guide_path: str | Path,
) -> tuple[dict[str, list[Requirement]], dict[str, dict], dict]:
    """Load everything keyed by canonical section key."""
    requirements = {
        sec: load_requirements(p) for sec, p in requirement_paths.items()
    }
    payloads = {sec: load_payload(p) for sec, p in payload_paths.items()}
    style = load_style_guide(style_guide_path)
    # Defensive: keep only known sections, in canonical order.
    requirements = {s: requirements[s] for s in SETTINGS.section_order if s in requirements}
    payloads = {s: payloads[s] for s in SETTINGS.section_order if s in payloads}
    return requirements, payloads, style


"""Evidence-tag registry (Layer 1).

Maps the 22-term `evidence_tags` vocabulary used by the requirement files to the
top-level payload tables that can satisfy them. This is the deterministic
backbone of requirement->evidence binding: where a tag resolves to a present
table, binding is exact and auditable; where it resolves to nothing, that is a
strong, reviewable MISSING signal (e.g. an 'insurance' requirement against a
payload with no insurance table -- exactly the synthetic-gap case).

The map is intentionally a hand-curated, reviewable table rather than a learned
matcher: an auditor can read it and verify the mapping policy in one screen.
"""

# tag -> candidate payload top-level keys (any section). The binder intersects
# these with the keys actually present in the section payload.
TAG_TO_PAYLOAD_KEYS: dict[str, list[str]] = {
    # Governance
    "governance_body": ["governance", "board_minutes"],
    "management_role": ["governance", "board_minutes"],
    "remuneration": ["governance"],
    # Risk
    "risk_process": ["climate_risk_register", "physical_risk_exposures"],
    "scenario_analysis": ["climate_scenarios", "resilience_assessment", "transition_plan"],
    # Strategy
    "strategy_decision_making": [
        "transition_plan", "climate_opportunities", "value_chain_map", "resilience_assessment",
    ],
    "business_model_value_chain": ["value_chain_map", "bank", "climate_opportunities"],
    "financial_effects": ["climate_financial_effects", "financial_summary"],
    # Metrics & targets
    "metrics": [
        "reporting_kpis", "scope1", "scope2", "scope3_travel", "scope3_categories",
        "financed_emissions", "targets",
    ],
    "targets": ["targets"],
    "ghg_emissions": [
        "scope1", "scope2", "scope3_travel", "scope3_categories",
        "financed_emissions", "ghg_methodology", "reporting_kpis",
    ],
    "scope_1": ["scope1"],
    "scope_2": ["scope2"],
    "scope_3": ["scope3_travel", "scope3_categories"],
    "financed_emissions": [
        "financed_emissions", "financed_emissions_equity", "financed_emissions_sovereign",
    ],
    "carbon_credits": ["carbon_credits", "internal_carbon_price"],
    "commercial_banking": ["financed_emissions", "financial_summary"],
    "asset_management": ["financed_emissions_equity"],
    # No insurance table exists in the BANK01 payloads -> deliberate gap signal.
    "insurance": [],
    # General requirements / connectivity
    "materiality": ["general_requirements_context", "metadata"],
    "connected_information": ["financial_summary", "reporting_kpis"],
    "source_guidance": ["metadata", "ghg_methodology"],
}

# Tables every section may draw on for entity context regardless of tags.
ALWAYS_AVAILABLE_KEYS: list[str] = ["bank"]


def candidate_keys_for_tags(tags: list[str], present_keys: set[str]) -> list[str]:
    """Resolve evidence tags to payload tables actually present in the section."""
    keys: list[str] = []
    for tag in tags:
        for k in TAG_TO_PAYLOAD_KEYS.get(tag, []):
            if k in present_keys and k not in keys:
                keys.append(k)
    return keys


"""Requirement -> evidence binder (Layer 1).

For each requirement, resolve its evidence tags to payload tables (registry),
then extract a compact, reviewable evidence bundle. This is deterministic; the
LLM only *confirms* sufficiency afterwards (classifier.py). The output feeds the
RequirementBinding contract.

The binder deliberately trims large tables (lists) to a representative slice so
the classifier/writer prompts stay bounded; the FULL resolved subtree is still
recorded by path for the audit trail.
"""

from typing import Any


_MAX_LIST_ITEMS = 6  # cap list evidence to keep prompts bounded


def _present_keys(payload: dict) -> set[str]:
    return set(payload.keys())


def _slice_value(value: Any) -> Any:
    """Bound list sizes for prompt economy; keep dicts whole."""
    if isinstance(value, list) and len(value) > _MAX_LIST_ITEMS:
        return value[:_MAX_LIST_ITEMS]
    return value


def build_evidence_bundle(
    requirement: Requirement, payload: dict
) -> tuple[dict[str, Any], list[EvidenceRef]]:
    """Return (evidence_bundle_for_prompt, evidence_refs_for_audit)."""
    present = _present_keys(payload)
    keys = candidate_keys_for_tags(requirement.evidence_tags, present)
    # Always include entity context, but only as background.
    for k in ALWAYS_AVAILABLE_KEYS:
        if k in present and k not in keys:
            keys.append(k)

    bundle: dict[str, Any] = {}
    refs: list[EvidenceRef] = []
    for k in keys:
        value = payload[k]
        bundle[k] = _slice_value(value)
        refs.append(
            EvidenceRef(path=k, value=_summarize(value), source_table=k)
        )
    return bundle, refs


def _summarize(value: Any) -> Any:
    """Compact representation stored in the audit ref (not the full subtree)."""
    if isinstance(value, list):
        return f"list[{len(value)}]"
    if isinstance(value, dict):
        return f"dict[{len(value)} keys]"
    return value


def has_evidence(requirement: Requirement, payload: dict) -> bool:
    """Fast deterministic pre-check: do any tags resolve to a present table?"""
    present = _present_keys(payload)
    return bool(candidate_keys_for_tags(requirement.evidence_tags, present))


"""Sufficiency classifier (Layer 1).

Produces the RequirementBinding contract. Policy:

1. Deterministic pre-gate: if no evidence tag resolves to a present payload
   table, the requirement is MISSING with NO LLM call (conservative + cheap).
2. Otherwise an LLM (extractor role, temperature 0) confirms whether the bundle
   SUPPORTED / PARTIAL / MISSING satisfies the requirement, returning structured
   JSON only. The model can never *invent* support: it sees only the bundle.

The decision is materialised as data in the binding, so even though an LLM
informs it, the result is auditable and reproducible.
"""


def _weight(req: Requirement) -> float:
    """Coverage weight. Mandatory requirements weigh full; scaled by quality."""
    base = 1.0 if req.mandatory else 0.5
    return base * max(0.1, req.requirement_quality_score)


def classify_requirement(
    requirement: Requirement,
    payload: dict,
    section_key: str,
    llm: LLMClient,
) -> RequirementBinding:
    weight = _weight(requirement)

    # 1. Deterministic MISSING pre-gate -------------------------------------
    if not has_evidence(requirement, payload):
        return RequirementBinding(
            requirement_id=requirement.requirement_id,
            section_key=section_key,
            status=SupportStatus.MISSING,
            evidence_refs=[],
            covered_elements=[],
            missing_elements=["all elements"],
            rationale="No evidence tag resolves to a payload table for this section.",
            weight=weight,
        )

    # 2. Build bundle + LLM sufficiency confirmation ------------------------
    bundle, refs = build_evidence_bundle(requirement, payload)
    raw = llm.complete(
        role="extractor",
        system=T.CLASSIFIER_SYSTEM,
        user=T.classifier_user(requirement.requirement_text, T.to_json(bundle)),
        json_mode=True,
    )
    parsed = parse_json_strict(raw)
    try:
        status = SupportStatus(parsed.get("status", "MISSING"))
    except ValueError:
        status = SupportStatus.MISSING

    return RequirementBinding(
        requirement_id=requirement.requirement_id,
        section_key=section_key,
        status=status,
        evidence_refs=refs if status != SupportStatus.MISSING else [],
        covered_elements=list(parsed.get("covered_elements", [])),
        missing_elements=list(parsed.get("missing_elements", [])),
        rationale=str(parsed.get("rationale", "")),
        weight=weight,
    )


def classify_section(
    requirements: list[Requirement],
    payload: dict,
    section_key: str,
    llm: LLMClient,
) -> list[RequirementBinding]:
    return [
        classify_requirement(r, payload, section_key, llm) for r in requirements
    ]


"""Gap channel + disclosable filter (Layer 1 -> Layer 2 boundary).

This module implements the central invariant in ONE place: the split between
what writers may see (disclosable) and what stays audit-only (gaps). Because the
two channels diverge here, deterministically, before any prompt is built, there
is no code path by which a MISSING requirement reaches a writer node.

The optional `data_gaps` block in the payload metadata is folded into the gap
ledger as an explicit synthetic-gap signal -- it informs the audit, never the
report.
"""


def split_channels(
    bindings: list[RequirementBinding],
    payload_metadata: dict | None = None,
    section_key: str = "section",
) -> tuple[list[RequirementBinding], list[GapEntry]]:
    """Return (disclosable_bindings, gap_entries).

    disclosable = SUPPORTED + PARTIAL (PARTIAL contributes its covered elements
    to the writer while its uncovered elements go to the gap ledger).
    """
    disclosable: list[RequirementBinding] = []
    gaps: list[GapEntry] = []

    for b in bindings:
        if b.status == SupportStatus.MISSING:
            gaps.append(
                GapEntry(
                    requirement_id=b.requirement_id,
                    section_key=b.section_key,
                    status=SupportStatus.MISSING,
                    missing_elements=b.missing_elements or ["all elements"],
                    reason=b.rationale or "No supporting evidence.",
                )
            )
            continue

        disclosable.append(b)
        if b.status == SupportStatus.PARTIAL and b.missing_elements:
            gaps.append(
                GapEntry(
                    requirement_id=b.requirement_id,
                    section_key=b.section_key,
                    status=SupportStatus.PARTIAL,
                    missing_elements=b.missing_elements,
                    reason="Partial coverage: listed elements lack evidence.",
                )
            )

    # Fold explicit synthetic data_gaps from payload metadata into the ledger.
    if payload_metadata:
        for dg in payload_metadata.get("data_gaps", []) or []:
            gaps.append(
                GapEntry(
                    requirement_id=f"data_gap::{dg.get('field', 'unknown')}",
                    section_key=section_key,
                    status=SupportStatus.MISSING,
                    missing_elements=[dg.get("field", "unknown")],
                    reason=str(dg.get("reason", "")),
                )
            )
    return disclosable, gaps

## 6. Style Rubric & Forbidden Phrases
Builds the scoring rubric **excluding** the style guide's missing-data rules, and defines the forbidden-phrase list (the status labels those rules would introduce) — the explicit conflict resolution.

In [7]:
"""Style rubric + forbidden-phrase policy.

CONFLICT RESOLUTION (made explicit for the reviewer)
----------------------------------------------------
The supplied global_style_guide.json contains `missing_data_language_rules` and
`disclosure_language_rules` that instruct the writer to STATE what is missing,
why, interim approaches, and to use status labels ('not currently available',
'under development', 'not reported for the period'). The controlling project
requirement is the opposite: missing data must be audit-only and must NEVER
appear in the report text.

We resolve the conflict in favour of the controlling requirement:
  * `missing_data_language_rules` are EXCLUDED from the scoring rubric, so a
    section is never penalised for omitting missing-data language; and
  * the status-label phrases those rules would introduce are ADDED to the
    forbidden-phrase gate, so they are actively rejected if they appear.

This keeps the fairness and audit-only guarantees intact regardless of the
style guide's wording.
"""

# Style-guide keys that are SAFE to use as scoring dimensions.
_SCORABLE_STYLE_KEYS = (
    "report_voice",
    "tone",
    "point_of_view",
    "paragraph_rules",
    "sentence_rules",
    "evidence_and_traceability_rules",
    "formatting_rules",
)

# Style-guide keys deliberately EXCLUDED from scoring (conflict with audit-only).
_EXCLUDED_STYLE_KEYS = (
    "missing_data_language_rules",
    # disclosure_language_rules contains status-label guidance; we keep its
    # general verbs but strip availability-label items via the forbidden list.
)

# Phrases that must never appear in the report (missing-data language).
FORBIDDEN_PHRASES: tuple[str, ...] = (
    "not available",
    "not currently available",
    "unavailable",
    "no data",
    "data is not",
    "data was not",
    "not disclosed",
    "not reported",
    "not reported for the period",
    "under development",
    "to be determined",
    "tbd",
    "n/a",
    "not applicable",
    "not material",
    "due to lack of",
    "lack of data",
    "insufficient data",
    "missing data",
    "incomplete data",
    "no information",
    "information was not provided",
    "could not be obtained",
    "pending",
    "placeholder",
    "not yet",
    "we do not have",
    "is not known",
)


def build_style_rubric(style_guide: dict) -> dict:
    """Return a rubric dict for the style judge, excluding missing-data rules."""
    rubric: dict = {}
    for key in _SCORABLE_STYLE_KEYS:
        if key in style_guide:
            rubric[key] = style_guide[key]
    # `do_not_do` is useful for style EXCEPT items about declaring missing data.
    if "do_not_do" in style_guide:
        rubric["do_not_do"] = [
            item
            for item in style_guide["do_not_do"]
            if "missing" not in item.lower() and "blank" not in item.lower()
        ]
    rubric["_scoring_dimensions"] = ["tone", "structure", "ifrs_register", "conciseness"]
    rubric["_excluded_for_audit_only"] = list(_EXCLUDED_STYLE_KEYS)
    return rubric

## 7. Validators
Deterministic gates first (number grounding, forbidden phrases), then LLM judges (claim entailment, style), assembled into one `ValidationReport` by the battery.

In [8]:
"""Numeric grounding validator (deterministic).

Every number, percentage, and year in the draft must match a value present in
the section evidence within tolerance. This is the strongest, cheapest
anti-hallucination lever: ungrounded numbers are caught exactly, without an LLM.

Matching tolerates rounding (the writer may render 1154.8131 as 1154.8) via a
relative tolerance, and recognises percentages and thousands separators.
"""

import re
from typing import Any


# Matches 1,154.81 / 1154.8 / 10.71% / 2024 / 48000
_NUM_RE = re.compile(r"(?<![\w.])(\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?\s?%?")


def _collect_numeric_values(evidence: Any, out: dict[float, str], path: str = "") -> None:
    if isinstance(evidence, dict):
        for k, v in evidence.items():
            _collect_numeric_values(v, out, f"{path}.{k}" if path else str(k))
    elif isinstance(evidence, list):
        for i, v in enumerate(evidence):
            _collect_numeric_values(v, out, f"{path}[{i}]")
    elif isinstance(evidence, bool):
        return
    elif isinstance(evidence, (int, float)):
        out[float(evidence)] = path
    elif isinstance(evidence, str):
        # numbers embedded in evidence strings count as grounded too
        for m in _NUM_RE.finditer(evidence):
            try:
                out[float(m.group(0).replace(",", "").replace("%", "").strip())] = path
            except ValueError:
                pass


def _match(value: float, evidence_values: dict[float, str]) -> tuple[bool, str | None, float | None]:
    rel = SETTINGS.thresholds.numeric_rel_tolerance
    abs_t = SETTINGS.thresholds.numeric_abs_tolerance
    for ev_val, ev_path in evidence_values.items():
        denom = max(abs(ev_val), abs(value), 1.0)
        if abs(ev_val - value) <= max(abs_t, rel * denom):
            return True, ev_path, ev_val
        # also accept rounded-to-1dp / integer rounding
        if round(ev_val, 1) == round(value, 1) or round(ev_val) == round(value):
            return True, ev_path, ev_val
    return False, None, None


def check_numbers(draft: str, evidence: Any) -> list[NumericCheck]:
    evidence_values: dict[float, str] = {}
    _collect_numeric_values(evidence, evidence_values)

    checks: list[NumericCheck] = []
    seen: set[str] = set()
    for m in _NUM_RE.finditer(draft):
        surface = m.group(0).strip()
        if surface in seen:
            continue
        seen.add(surface)
        is_pct = surface.endswith("%")
        try:
            normalized = float(surface.replace(",", "").replace("%", "").strip())
        except ValueError:
            continue
        grounded, path, matched = _match(normalized, evidence_values)
        checks.append(
            NumericCheck(
                surface=surface,
                normalized=normalized,
                unit="%" if is_pct else None,
                grounded=grounded,
                matched_path=path,
                matched_value=matched,
            )
        )
    return checks


"""Forbidden-phrase validator (deterministic).

Third layer of the no-missing-data-language defense. Even though the writer
never sees gaps (structural) and is told not to editorialise about coverage
(prompt), this gate deterministically rejects any draft that slips in
availability/absence language or status labels.
"""

import re


_CONTEXT = 40


def check_phrases(draft: str) -> list[PhraseHit]:
    low = draft.lower()
    hits: list[PhraseHit] = []
    for phrase in FORBIDDEN_PHRASES:
        for m in re.finditer(r"(?<![\w])" + re.escape(phrase) + r"(?![\w])", low):
            start = max(0, m.start() - _CONTEXT)
            end = min(len(draft), m.end() + _CONTEXT)
            hits.append(PhraseHit(phrase=phrase, context=draft[start:end].strip()))
    return hits


"""Claim entailment validator (LLM-as-judge).

Decomposes the draft into atomic claims (extractor role) then judges each
against the evidence (judge role -- the stronger, *different* model, to reduce
self-preference bias relative to the writer). A claim that is plausible but not
present in the evidence is NOT_ENTAILED and fails the gate.
"""


def check_claims(draft: str, evidence: dict, llm: LLMClient) -> list[ClaimCheck]:
    # 1. extract atomic claims
    raw_claims = llm.complete(
        role="extractor",
        system=T.CLAIM_EXTRACT_SYSTEM,
        user=T.claim_extract_user(draft),
        json_mode=True,
    )
    claims = parse_json_strict(raw_claims).get("claims", [])
    if not claims:
        return []

    # 2. judge entailment against evidence
    raw_verdicts = llm.complete(
        role="judge",
        system=T.ENTAILMENT_SYSTEM,
        user=T.entailment_user(T.to_json({"claims": claims}), T.to_json(evidence)),
        json_mode=True,
    )
    results = parse_json_strict(raw_verdicts).get("results", [])

    checks: list[ClaimCheck] = []
    for r in results:
        try:
            verdict = ClaimVerdict(r.get("verdict", "NOT_ENTAILED"))
        except ValueError:
            verdict = ClaimVerdict.NOT_ENTAILED
        checks.append(
            ClaimCheck(
                claim=str(r.get("claim", "")),
                verdict=verdict,
                evidence_ref=r.get("evidence_ref"),
                note=str(r.get("note", "")),
            )
        )
    return checks


"""Style validator (LLM-as-judge).

Scores the draft against the rubric built in style.rubric (which already
excludes missing-data rules). Explicit per-dimension scoring keeps the judgement
reproducible and lets a reviewer see *why* a style score landed where it did.
"""


def score_style(draft: str, rubric: dict, llm: LLMClient) -> StyleScore:
    raw = llm.complete(
        role="judge",
        system=T.STYLE_SYSTEM,
        user=T.style_user(draft, T.to_json(rubric)),
        json_mode=True,
    )
    parsed = parse_json_strict(raw)
    dims = {k: float(v) for k, v in parsed.get("dimensions", {}).items()}
    overall = sum(dims.values()) / len(dims) if dims else 0.0
    return StyleScore(dimensions=dims, overall=overall, notes=str(parsed.get("notes", "")))


"""Validator battery.

Runs all four validators and assembles a ValidationReport. Order is intentional:
the cheap deterministic gates (numbers, phrases) run first; the LLM judges
(claims, style) run after. The ValidationReport's gate properties decide pass/fail.
"""


def run_validation(
    draft: str,
    evidence: dict,
    rubric: dict,
    llm: LLMClient,
) -> ValidationReport:
    numeric_checks = check_numbers(draft, evidence)
    phrase_hits = check_phrases(draft)
    claim_checks = check_claims(draft, evidence, llm)
    style = score_style(draft, rubric, llm)
    return ValidationReport(
        numeric_checks=numeric_checks,
        claim_checks=claim_checks,
        phrase_hits=phrase_hits,
        style=style,
    )

## 8. Scoring
Three independent sub-scores; coverage uses the **disclosable denominator** so synthetic gaps cannot lower it. Integrity is a min + boolean, never averaged away.

In [9]:
"""Scoring (deterministic).

Three INDEPENDENT sub-scores per section, never collapsed into one number until
report level -- and even then integrity stays a separate pass/fail gate, never
traded against coverage or style.

The fairness guarantee lives in the denominator: coverage is computed over the
DISCLOSABLE requirements only. MISSING requirements are removed before the ratio,
so a section that fully covers everything it has evidence for scores 1.0 even
when half its IFRS datapoints have no synthetic data. Synthetic gaps are thus
mathematically incapable of lowering any score.
"""


# PARTIAL contributes a fraction of its weight to coverage.
_PARTIAL_CREDIT = 0.6


def score_section(
    bindings: list[RequirementBinding],
    validation: ValidationReport,
) -> SectionScore:
    total = len(bindings)
    disclosable = [b for b in bindings if b.is_disclosable]
    excluded = total - len(disclosable)

    # Coverage over the DISCLOSABLE denominator only.
    if disclosable:
        earned = 0.0
        possible = 0.0
        for b in disclosable:
            possible += b.weight
            if b.status == SupportStatus.SUPPORTED:
                earned += b.weight
            elif b.status == SupportStatus.PARTIAL:
                earned += b.weight * _PARTIAL_CREDIT
        coverage = earned / possible if possible else 0.0
    else:
        # Nothing disclosable: coverage is undefined, reported as 1.0 (the
        # section correctly discloses the empty set) rather than 0.0, so an
        # all-synthetic-gap section is not penalised.
        coverage = 1.0

    return SectionScore(
        coverage=round(coverage, 4),
        integrity=round(validation.integrity_score, 4),
        style=round(validation.style.overall, 4),
        disclosable_count=len(disclosable),
        total_requirements=total,
        excluded_missing=excluded,
    )


def aggregate_report_score(section_scores: dict[str, SectionScore]) -> dict:
    """Report-level rollup. Coverage/style weighted by disclosable_count.

    integrity is reported as a min across sections AND as a hard boolean; it is
    never averaged into a single headline number that could mask a failure.
    """
    if not section_scores:
        return {"coverage": 0.0, "style": 0.0, "integrity_min": 1.0, "integrity_ok": True}

    total_w = sum(s.disclosable_count for s in section_scores.values()) or 1
    coverage = sum(s.coverage * s.disclosable_count for s in section_scores.values()) / total_w
    style = sum(s.style * s.disclosable_count for s in section_scores.values()) / total_w
    integrity_min = min(s.integrity for s in section_scores.values())

    return {
        "coverage": round(coverage, 4),
        "style": round(style, 4),
        "integrity_min": round(integrity_min, 4),
        "integrity_ok": integrity_min >= 0.999,
        "per_section": {k: v.model_dump() for k, v in section_scores.items()},
    }

## 9. Assembly · 10. Consistency · 11. Audit (Layer 3)
Deterministic ordered assembly; cross-section numeric reconciliation + a coherence judge; and the audit bundle / gap report / summary writers. (`build_run_metadata` hashes the prompt strings for reproducibility.)

In [10]:
"""Report assembly (Layer 3, deterministic).

Concatenates accepted section drafts in canonical IFRS order with front matter
and a table of contents. No LLM here, so the final document is a reproducible
function of the accepted section drafts.
"""

from datetime import date


def assemble_report(
    sections: dict[str, SectionState],
    settings: Settings,
    bank_id: str,
) -> str:
    parts: list[str] = []
    parts.append("# Sustainability-related Financial Disclosures")
    parts.append(f"_Prepared in accordance with IFRS S1 and IFRS S2 — entity {bank_id}_")
    parts.append(f"_Generated: {date.today().isoformat()}_")
    parts.append("")

    # Table of contents (canonical order).
    parts.append("## Contents")
    for i, key in enumerate(settings.section_order, start=1):
        if key in sections:
            parts.append(f"{i}. {settings.title_for(key)}")
    parts.append("")

    # Sections in canonical order.
    for i, key in enumerate(settings.section_order, start=1):
        sec = sections.get(key)
        if not sec or not sec.draft.strip():
            continue
        body = sec.draft.strip()
        # Normalise: ensure a numbered top heading, drop any writer-added H2 title.
        body = _strip_leading_heading(body)
        parts.append(f"## {i}. {sec.title}")
        if sec.status == SectionStatus.NEEDS_HUMAN_REVIEW:
            parts.append("> _[internal: section flagged for human review]_")
        parts.append("")
        parts.append(body)
        parts.append("")
    return "\n".join(parts).strip() + "\n"


def _strip_leading_heading(body: str) -> str:
    lines = body.splitlines()
    if lines and lines[0].lstrip().startswith("#"):
        return "\n".join(lines[1:]).lstrip("\n")
    return body


"""Cross-section consistency (Layer 3).

Two parts:
  1. Deterministic numeric reconciliation -- the valuable one. Because every
     number is already grounded to an evidence path, a figure that appears in
     more than one section must resolve to the SAME evidence value. Divergence
     is flagged precisely.
  2. LLM coherence judge -- terminology/narrative consistency (a governance body
     named consistently, no contradictions).

Returns a structured report; the graph decides whether to route an offending
section back for targeted re-revision.
"""

from collections import defaultdict


def reconcile_numbers(sections: dict[str, SectionState]) -> list[dict]:
    """Flag numbers that map to different evidence values across sections."""
    # value(rounded) -> {section -> matched_path}
    by_value: dict[float, dict[str, str]] = defaultdict(dict)
    for key, sec in sections.items():
        if not sec.validation:
            continue
        for nc in sec.validation.numeric_checks:
            if nc.grounded and nc.matched_value is not None:
                by_value[round(nc.matched_value, 4)][key] = nc.matched_path or ""

    # Detect the same surfaced figure grounded to different evidence paths.
    issues: list[dict] = []
    path_to_values: dict[str, set[float]] = defaultdict(set)
    for value, sec_map in by_value.items():
        for path in sec_map.values():
            path_to_values[path].add(value)
    for path, values in path_to_values.items():
        if len(values) > 1:
            issues.append(
                {"type": "numeric_divergence", "evidence_path": path,
                 "values": sorted(values)}
            )
    return issues


def judge_coherence(report_text: str, llm: LLMClient) -> list[dict]:
    raw = llm.complete(
        role="judge",
        system=T.CONSISTENCY_SYSTEM,
        user=T.consistency_user(report_text),
        json_mode=True,
    )
    return parse_json_strict(raw).get("inconsistencies", [])


def run_consistency(
    sections: dict[str, SectionState], report_text: str, llm: LLMClient
) -> dict:
    numeric_issues = reconcile_numbers(sections)
    coherence_issues = judge_coherence(report_text, llm)
    return {
        "numeric_issues": numeric_issues,
        "coherence_issues": coherence_issues,
        "consistent": not numeric_issues and not coherence_issues,
    }


"""Audit export (Layer 3).

Writes machine-readable JSON plus a human-readable Markdown summary per run.
The score breakdown shows the DISCLOSABLE denominator explicitly so a reviewer
can see *why* a synthetic gap did not lower the score. Run metadata (model
deployments, prompt module hash, evidence snapshot hash, timestamp) makes every
report reproducible and defensible.
"""

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path


def _hash_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


def build_run_metadata(payloads: dict, model_cfg) -> dict:
    snapshot = json.dumps(payloads, sort_keys=True, default=str)
    prompt_src = "\n".join(sorted(v for v in vars(T).values() if isinstance(v, str)))
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "models": {
            "writer": model_cfg.writer_deployment,
            "extractor": model_cfg.extractor_deployment,
            "judge": model_cfg.judge_deployment,
            "reviser": model_cfg.reviser_deployment,
        },
        "prompt_module_hash": _hash_text(prompt_src),
        "evidence_snapshot_hash": _hash_text(snapshot),
    }


def export_audit(
    audit: AuditBundle,
    report_score: dict,
    consistency_report: dict,
    out_dir: str | Path,
) -> dict[str, str]:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    bundle_json = {
        "run_metadata": audit.run_metadata,
        "score": report_score,
        "consistency": consistency_report,
        "binding_ledger": [b.model_dump() for b in audit.binding_ledger],
        "gap_ledger": [g.model_dump() for g in audit.gap_ledger],
        "claim_verdicts": {
            k: [c.model_dump() for c in v] for k, v in audit.claim_verdicts.items()
        },
        "numeric_ledger": {
            k: [n.model_dump() for n in v] for k, v in audit.numeric_ledger.items()
        },
        "revision_history": audit.revision_history,
    }
    json_path = out / "audit_bundle.json"
    json_path.write_text(json.dumps(bundle_json, indent=2, default=str), encoding="utf-8")

    gap_path = out / "gap_report.json"
    gap_path.write_text(
        json.dumps([g.model_dump() for g in audit.gap_ledger], indent=2, default=str),
        encoding="utf-8",
    )

    summary_path = out / "audit_summary.md"
    summary_path.write_text(_render_summary(audit, report_score, consistency_report), encoding="utf-8")

    return {
        "audit_bundle": str(json_path),
        "gap_report": str(gap_path),
        "audit_summary": str(summary_path),
    }


def _render_summary(audit: AuditBundle, score: dict, consistency: dict) -> str:
    lines = ["# Audit Summary", ""]
    md = audit.run_metadata
    lines.append(f"- Generated: {md.get('timestamp_utc', 'n/a')}")
    lines.append(f"- Evidence snapshot: `{md.get('evidence_snapshot_hash', 'n/a')}`")
    lines.append(f"- Prompt module: `{md.get('prompt_module_hash', 'n/a')}`")
    lines.append("")
    lines.append("## Scores (disclosable denominator)")
    lines.append(f"- Report coverage: {score.get('coverage')}")
    lines.append(f"- Report style: {score.get('style')}")
    lines.append(f"- Integrity (min across sections): {score.get('integrity_min')} "
                 f"-> {'PASS' if score.get('integrity_ok') else 'FAIL'}")
    lines.append("")
    lines.append("| Section | Coverage | Integrity | Style | Disclosable | Excluded (missing) |")
    lines.append("|---|---|---|---|---|---|")
    for k, s in (score.get("per_section") or {}).items():
        lines.append(
            f"| {k} | {s['coverage']} | {s['integrity']} | {s['style']} | "
            f"{s['disclosable_count']} | {s['excluded_missing']} |"
        )
    lines.append("")
    lines.append(f"## Gaps (audit-only): {len(audit.gap_ledger)} entries")
    lines.append("_These are recorded for traceability and never appear in the report._")
    lines.append("")
    lines.append(f"## Consistency: {'CONSISTENT' if consistency.get('consistent') else 'ISSUES FOUND'}")
    if consistency.get("numeric_issues"):
        lines.append(f"- Numeric divergences: {len(consistency['numeric_issues'])}")
    if consistency.get("coherence_issues"):
        lines.append(f"- Coherence issues: {len(consistency['coherence_issues'])}")
    return "\n".join(lines) + "\n"

## 12. Section Subgraph (Layer 2)
The per-section LangGraph: `prepare_context → generate → validate → [accept | revise→validate | flag_human]`. LLM/rubric are injected via closures, not state, so the graph stays checkpointable.

In [11]:
"""Section subgraph (Layer 2).

One parameterised subgraph, instantiated per section. Topology:

    prepare_context -> generate -> validate -> [route]
                          ^                        |
                          |                    (fail & budget left)
                          +------- revise <--------+
                                                   |
                                              (pass)|-> score -> accept
                                       (budget spent)-> flag_human -> score -> accept

Non-serialisable dependencies (the LLM client, rubric) are injected via closures
in the factory rather than placed in state, so the graph remains checkpointable.
Writers only ever receive `disclosable` bindings and the section evidence bundle;
gap information never enters this subgraph.
"""

from typing import Annotated, Optional, TypedDict

from langgraph.graph import END, START, StateGraph


class SectionGraphState(TypedDict, total=False):
    section_key: str
    title: str
    bindings: list[RequirementBinding]   # disclosable only
    payload: dict
    evidence: dict                        # section-level bundle (set in prepare)
    requirement_briefs: str
    draft: str
    validation: Optional[ValidationReport]
    score: Optional[SectionScore]
    revision_count: int
    status: SectionStatus
    revision_log: Annotated[list[str], lambda a, b: (a or []) + (b or [])]


def _section_evidence(bindings: list[RequirementBinding], payload: dict) -> dict:
    """Union of payload tables referenced by this section's disclosable bindings."""
    keys: list[str] = []
    for b in bindings:
        for ref in b.evidence_refs:
            if ref.source_table and ref.source_table not in keys:
                keys.append(ref.source_table)
    return {k: payload[k] for k in keys if k in payload}


def _requirement_briefs(bindings: list[RequirementBinding]) -> str:
    lines = []
    for b in bindings:
        covered = ", ".join(b.covered_elements[:6]) or "see evidence"
        lines.append(f"- [{b.requirement_id}] cover: {covered}")
    return "\n".join(lines)


def make_section_graph(llm: LLMClient, settings: Settings, rubric: dict):
    """Factory: returns a compiled section subgraph closing over llm/rubric."""
    th = settings.thresholds

    def prepare_context(state: SectionGraphState) -> dict:
        bindings = state["bindings"]
        evidence = _section_evidence(bindings, state["payload"])
        return {
            "evidence": evidence,
            "requirement_briefs": _requirement_briefs(bindings),
            "revision_count": 0,
            "status": SectionStatus.PENDING,
        }

    def generate(state: SectionGraphState) -> dict:
        draft = llm.complete(
            role="writer",
            system=T.WRITER_SYSTEM,
            user=T.writer_user(
                state["title"],
                T.to_json(rubric),
                state["requirement_briefs"],
                T.to_json(state["evidence"]),
            ),
        )
        return {"draft": draft, "status": SectionStatus.DRAFTED}

    def validate(state: SectionGraphState) -> dict:
        report = run_validation(state["draft"], state["evidence"], rubric, llm)
        return {"validation": report}

    def revise(state: SectionGraphState) -> dict:
        failures = "\n".join(f"- {f}" for f in state["validation"].failure_summary())
        fixed = llm.complete(
            role="reviser",
            system=T.REVISER_SYSTEM,
            user=T.reviser_user(state["draft"], failures, T.to_json(state["evidence"])),
        )
        n = state.get("revision_count", 0) + 1
        return {
            "draft": fixed,
            "revision_count": n,
            "revision_log": [f"revision {n}: fixed {len(state['validation'].failure_summary())} issue(s)"],
        }

    def flag_human(state: SectionGraphState) -> dict:
        return {
            "status": SectionStatus.NEEDS_HUMAN_REVIEW,
            "revision_log": ["exhausted revision budget; flagged for human review"],
        }

    def accept(state: SectionGraphState) -> dict:
        # Scoring is done at the runner level where ALL bindings (incl. MISSING)
        # are visible; here we only finalise status.
        status = state.get("status")
        if status != SectionStatus.NEEDS_HUMAN_REVIEW:
            status = SectionStatus.ACCEPTED
        return {"status": status}

    def route_after_validate(state: SectionGraphState) -> str:
        report = state["validation"]
        if report.hard_gates_pass(th.style_min):
            return "accept"
        if state.get("revision_count", 0) < th.max_revisions:
            return "revise"
        return "flag_human"

    g = StateGraph(SectionGraphState)
    g.add_node("prepare_context", prepare_context)
    g.add_node("generate", generate)
    g.add_node("validate", validate)
    g.add_node("revise", revise)
    g.add_node("flag_human", flag_human)
    g.add_node("accept", accept)

    g.add_edge(START, "prepare_context")
    g.add_edge("prepare_context", "generate")
    g.add_edge("generate", "validate")
    g.add_conditional_edges(
        "validate", route_after_validate,
        {"accept": "accept", "revise": "revise", "flag_human": "flag_human"},
    )
    g.add_edge("revise", "validate")
    g.add_edge("flag_human", "accept")
    g.add_edge("accept", END)
    return g.compile()

## 13. Global Nodes
Layer-1 `bind` (classify all sections + split channels), the per-section runner factory (fan-out; empty sections score fairly), assemble, consistency, and audit collation. Section scoring happens here, where ALL bindings (incl. MISSING) are visible.

In [12]:
"""Global (top-level) graph nodes.

Layer 1 bind/classify -> fan-out to five section runners -> Layer 3 assemble,
consistency, audit export. The section runners invoke the compiled section
subgraph; they run concurrently because they share the `bind` predecessor and
`assemble` successor and write disjoint keys into the merge-reduced `sections`.

All inter-node handoffs use DECLARED state channels (LangGraph drops undeclared
keys), so the dataflow is explicit and checkpointable.
"""


def make_global_nodes(llm: LLMClient, settings: Settings, rubric: dict, section_graph):
    # --- Layer 1: classify every requirement, split channels --------------
    def bind(state: GraphState) -> dict:
        bindings_by_section: dict[str, list] = {}
        disclosable_by_section: dict[str, list] = {}
        audit = AuditBundle()

        for key in settings.section_order:
            if key not in state["requirements"]:
                continue
            reqs = state["requirements"].get(key, [])
            payload = state["payloads"].get(key, {})
            bindings = classify_section(reqs, payload, key, llm)
            disclosable, gaps = split_channels(bindings, payload.get("metadata"), key)
            bindings_by_section[key] = bindings
            disclosable_by_section[key] = disclosable
            audit.binding_ledger.extend(bindings)
            audit.gap_ledger.extend(gaps)

        sections = {
            key: SectionState(section_key=key, title=settings.title_for(key))
            for key in settings.section_order
            if key in state["requirements"]
        }
        return {
            "bindings": bindings_by_section,
            "disclosable": disclosable_by_section,
            "sections": sections,
            "audit": audit,
        }

    # --- Layer 2: one runner per section (fan-out) ------------------------
    def make_section_runner(section_key: str):
        def runner(state: GraphState) -> dict:
            disclosable = state.get("disclosable", {}).get(section_key, [])
            payload = state["payloads"].get(section_key, {})
            all_bindings = state["bindings"][section_key]

            if not disclosable:
                # Nothing disclosable -> empty, fully-fair section (coverage 1.0).
                sec = SectionState(
                    section_key=section_key,
                    title=settings.title_for(section_key),
                    status=SectionStatus.ACCEPTED,
                    draft="",
                    score=score_section(all_bindings, ValidationReport()),
                )
                return {"sections": {section_key: sec}}

            result = section_graph.invoke(
                {
                    "section_key": section_key,
                    "title": settings.title_for(section_key),
                    "bindings": disclosable,
                    "payload": payload,
                }
            )
            validation = result.get("validation") or ValidationReport()
            sec = SectionState(
                section_key=section_key,
                title=settings.title_for(section_key),
                draft=result.get("draft", ""),
                status=result.get("status", SectionStatus.ACCEPTED),
                revision_count=result.get("revision_count", 0),
                validation=result.get("validation"),
                # Score over ALL bindings (incl. MISSING) so excluded counts are
                # accurate, while coverage still uses the disclosable denominator.
                score=score_section(all_bindings, validation),
                revision_log=result.get("revision_log", []),
            )
            return {"sections": {section_key: sec}}

        return runner

    # --- Layer 3: assemble + collate audit --------------------------------
    def assemble(state: GraphState) -> dict:
        sections = state["sections"]
        report = assemble_report(sections, settings, state.get("bank_id", "ENTITY"))

        audit = state["audit"]
        for key, sec in sections.items():
            if sec.validation:
                audit.claim_verdicts[key] = sec.validation.claim_checks
                audit.numeric_ledger[key] = sec.validation.numeric_checks
            if sec.revision_log:
                audit.revision_history[key] = sec.revision_log
        return {"final_report": report, "audit": audit}

    def consistency(state: GraphState) -> dict:
        rep = run_consistency(state["sections"], state["final_report"], llm)
        return {"consistency_report": rep}

    def finalize_audit(state: GraphState) -> dict:
        section_scores = {
            k: s.score for k, s in state["sections"].items() if s.score is not None
        }
        report_score = aggregate_report_score(section_scores)
        audit = state["audit"]
        audit.run_metadata["report_score"] = report_score
        return {"audit": audit, "report_score": report_score}

    return {
        "bind": bind,
        "make_section_runner": make_section_runner,
        "assemble": assemble,
        "consistency": consistency,
        "finalize_audit": finalize_audit,
    }

## 14. Graph & Pipeline
Top graph: `bind → fan-out run_<section>×5 → assemble → consistency → finalize_audit`. The `Pipeline` facade compiles once and runs many.

In [13]:
"""Graph construction + Pipeline facade.

Builds the top-level StateGraph (bind -> fan-out section runners -> assemble ->
consistency -> finalize_audit) and exposes a small Pipeline class that compiles
once and runs end to end. The LLM client and settings are injected, so the same
graph runs with the offline MockLLM or the AzureOpenAILLM unchanged.
"""

from typing import Optional

from langgraph.graph import END, START, StateGraph


def build_main_graph(
    llm: LLMClient,
    style_guide: dict,
    settings: Settings = SETTINGS,
    checkpointer=None,
):
    rubric = build_style_rubric(style_guide)
    section_graph = make_section_graph(llm, settings, rubric)
    nodes = make_global_nodes(llm, settings, rubric, section_graph)

    g = StateGraph(GraphState)
    g.add_node("bind", nodes["bind"])
    g.add_node("assemble", nodes["assemble"])
    g.add_node("consistency", nodes["consistency"])
    g.add_node("finalize_audit", nodes["finalize_audit"])

    g.add_edge(START, "bind")
    # Fan-out: one runner node per canonical section, all between bind/assemble.
    for key in settings.section_order:
        node_name = f"run_{key}"
        g.add_node(node_name, nodes["make_section_runner"](key))
        g.add_edge("bind", node_name)
        g.add_edge(node_name, "assemble")

    g.add_edge("assemble", "consistency")
    g.add_edge("consistency", "finalize_audit")
    g.add_edge("finalize_audit", END)

    return g.compile(checkpointer=checkpointer)


class Pipeline:
    """Compile-once, run-many facade."""

    def __init__(
        self,
        llm: LLMClient,
        style_guide: dict,
        settings: Settings = SETTINGS,
        checkpointer=None,
    ) -> None:
        self.settings = settings
        self.style_guide = style_guide
        self.graph = build_main_graph(llm, style_guide, settings, checkpointer)

    def run(
        self,
        requirements: dict,
        payloads: dict,
        bank_id: str = "ENTITY",
        config: Optional[dict] = None,
    ) -> GraphState:
        initial: GraphState = {
            "requirements": requirements,
            "payloads": payloads,
            "style_guide": self.style_guide,
            "bank_id": bank_id,
        }
        return self.graph.invoke(initial, config=config or {})

## 15. Run End-to-End
Loads the uploaded BANK01 data, runs the full pipeline with the offline `MockLLM`, writes the report + audit artifacts, and prints the score summary.

To run live: set `AZURE_OPENAI_ENDPOINT` / `AZURE_OPENAI_API_KEY` and replace `MockLLM()` with `AzureOpenAILLM()`.

In [14]:
UPLOADS = "/mnt/user-data/uploads"   # folder with the requirement files, payloads, style guide
OUT_DIR = "ifrs_output"

def resolve_paths(base: Path):
    req = {
        "general_requirements": base / "general_requirements_requirements.json",
        "governance": base / "governance_requirements.json",
        "strategy": base / "strategy_requirements.json",
        "risk_management": base / "risk_management_requirements.json",
        "metrics_and_targets": base / "metrics_and_targets_requirements.json",
    }
    pay = {
        "general_requirements": base / "payload_BANK01_general_requirements.json",
        "governance": base / "payload_BANK01_governance.json",
        "strategy": base / "payload_BANK01_strategy.json",
        "risk_management": base / "payload_BANK01_risk_management.json",
        "metrics_and_targets": base / "payload_BANK01_metrics_targets.json",
    }
    return req, pay, base / "global_style_guide.json"

base = Path(UPLOADS)
req_paths, pay_paths, style_path = resolve_paths(base)
requirements, payloads, style_guide = load_all(req_paths, pay_paths, style_path)

llm = MockLLM()                      # <-- swap for AzureOpenAILLM() to run live
pipeline = Pipeline(llm, style_guide, SETTINGS)
final = pipeline.run(requirements, payloads, bank_id="BANK01")

audit = final['audit']
audit.run_metadata.update(build_run_metadata(payloads, SETTINGS.model))
out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)
(out / "BANK01_sustainability_report.md").write_text(final["final_report"], encoding="utf-8")
paths = export_audit(audit, final['report_score'], final['consistency_report'], out)

score = final['report_score']
print("coverage=", score["coverage"], " style=", score["style"],
      " integrity_min=", score["integrity_min"], "PASS" if score["integrity_ok"] else "FAIL")
for k, s in score['per_section'].items():
    print(f"  {k:24s} cov={s['coverage']:.3f} int={s['integrity']:.3f} sty={s['style']:.3f} "
          f"disclosable={s['disclosable_count']:>3} excluded_missing={s['excluded_missing']:>3}")
print('gap ledger (audit-only):', len(audit.gap_ledger), 'entries')
print('consistency:', 'CONSISTENT' if final['consistency_report']['consistent'] else 'ISSUES')
print('artifacts:', paths)

coverage= 1.0  style= 0.8625  integrity_min= 1.0 PASS
  general_requirements     cov=1.000 int=1.000 sty=0.863 disclosable= 83 excluded_missing= 25
  governance               cov=1.000 int=1.000 sty=0.863 disclosable= 14 excluded_missing=  1
  strategy                 cov=1.000 int=1.000 sty=0.863 disclosable= 60 excluded_missing= 10
  risk_management          cov=1.000 int=1.000 sty=0.863 disclosable= 16 excluded_missing=  1
  metrics_and_targets      cov=1.000 int=1.000 sty=0.863 disclosable=148 excluded_missing=  3
gap ledger (audit-only): 60 entries
consistency: CONSISTENT
artifacts: {'audit_bundle': 'ifrs_output/audit_bundle.json', 'gap_report': 'ifrs_output/gap_report.json', 'audit_summary': 'ifrs_output/audit_summary.md'}


## 16. Verify the Guarantees
Independent checks: no forbidden phrase survives in the final report, every number is grounded, and the MISSING requirements (incl. the insurance-only ones) are in the audit-only gap ledger.

In [15]:
report = (Path(OUT_DIR) / "BANK01_sustainability_report.md").read_text()
print('forbidden-phrase hits in final report:', len(check_phrases(report)))
ungrounded = sum(1 for v in audit.numeric_ledger.values() for c in v if not c.grounded)
total_nums = sum(len(v) for v in audit.numeric_ledger.values())
print(f'numbers checked={total_nums} ungrounded={ungrounded}')
missing = [g for g in audit.gap_ledger if g.status.value == 'MISSING' and not g.requirement_id.startswith('data_gap::')]
print('requirement-level MISSING (audit-only):', len(missing))
print('report preview:\n')
print('\n'.join(report.splitlines()[:16]))

forbidden-phrase hits in final report: 0
numbers checked=40 ungrounded=0
requirement-level MISSING (audit-only): 40
report preview:

# Sustainability-related Financial Disclosures
_Prepared in accordance with IFRS S1 and IFRS S2 — entity BANK01_
_Generated: 2026-06-29_

## Contents
1. General Requirements
2. Governance
3. Strategy
4. Risk Management
5. Metrics and Targets

## 1. General Requirements

This section sets out the general requirements disclosures supported by the available evidence for the reporting entity.

- The summary id is FIN-BANK01-2022.
